In [1]:
import numpy as np
import pandas as pd
import pickle
import joblib
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Load tabular pipeline
with open('data/processed/tabular_pipeline_v2.pkl', 'rb') as f:
    pipeline_data = pickle.load(f)

X_train = pipeline_data['X_train']
y_train = pipeline_data['y_train']
X_val   = pipeline_data['X_val']
y_val   = pipeline_data['y_val']
X_test  = pipeline_data['X_test']
y_test  = pipeline_data['y_test']

print("Data loaded!")
print(f"Tabular train: {X_train.shape}")

Data loaded!
Tabular train: (6800, 9)


In [2]:
# Smaller grid to avoid RAM crash
param_combinations = [
    {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1},
    {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05},
    {'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.05},
]

best_acc = 0
best_params = None
best_model = None

print("Testing parameter combinations...\n")

for params in param_combinations:
    model = xgb.XGBClassifier(
        **params,
        subsample=0.8,
        random_state=42,
        eval_metric='mlogloss',
        verbosity=0
    )
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              verbose=False)
    
    val_acc = accuracy_score(y_val, model.predict(X_val)) * 100
    test_acc = accuracy_score(y_test, model.predict(X_test)) * 100
    print(f"Params: {params}")
    print(f"  Val Acc: {val_acc:.2f}% | Test Acc: {test_acc:.2f}%\n")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_params = params
        best_model = model

print(f"Best params: {best_params}")
print(f"Best val accuracy: {best_acc:.2f}%")

Testing parameter combinations...

Params: {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1}
  Val Acc: 99.58% | Test Acc: 99.70%

Params: {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}
  Val Acc: 99.58% | Test Acc: 99.65%

Params: {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05}
  Val Acc: 99.58% | Test Acc: 99.65%

Params: {'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.05}
  Val Acc: 99.42% | Test Acc: 99.60%

Best params: {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1}
Best val accuracy: 99.58%


In [3]:
# Save the best tuned model
joblib.dump(best_model, 'models/xgboost_tuned.pkl')
print("Tuned XGBoost model saved to models/xgboost_tuned.pkl")

# Compare all models so far
print("\nModel comparison:")
print(f"  Original XGBoost (Day 6):     99.50%")
print(f"  Retrained XGBoost (Day 8):    99.50%")
print(f"  Tuned XGBoost (Day 15):       99.70%")
print(f"\nBest params: {best_params}")

Tuned XGBoost model saved to models/xgboost_tuned.pkl

Model comparison:
  Original XGBoost (Day 6):     99.50%
  Retrained XGBoost (Day 8):    99.50%
  Tuned XGBoost (Day 15):       99.70%

Best params: {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1}


In [4]:
print("Model Comparison Summary")
print("=" * 50)
print(f"\nResNet50:")
print(f"  Parameters:    23,538,767")
print(f"  Test Accuracy: 99.74%")
print(f"  Training time: ~2 hours on CPU")

print(f"\nEfficientNet-B3 (theoretical):")
print(f"  Parameters:    12,233,232 (48% smaller)")
print(f"  Expected Acc:  Similar or slightly better")
print(f"  Training time: ~1.5 hours on CPU")

print(f"\nDecision: Keep ResNet50 — already at 99.74%")
print(f"EfficientNet would be worth trying with a GPU")

print(f"\nFinal tuned models:")
print(f"  Image model:    ResNet50 balanced  → 99.74%")
print(f"  Tabular model:  XGBoost tuned      → 99.70%")

Model Comparison Summary

ResNet50:
  Parameters:    23,538,767
  Test Accuracy: 99.74%
  Training time: ~2 hours on CPU

EfficientNet-B3 (theoretical):
  Parameters:    12,233,232 (48% smaller)
  Expected Acc:  Similar or slightly better
  Training time: ~1.5 hours on CPU

Decision: Keep ResNet50 — already at 99.74%
EfficientNet would be worth trying with a GPU

Final tuned models:
  Image model:    ResNet50 balanced  → 99.74%
  Tabular model:  XGBoost tuned      → 99.70%


In [5]:
new_api = open('src/api.py').read().replace(
    'SCALER = joblib.load("models/tabular_scaler_v2.pkl")',
    '''SCALER = joblib.load("models/tabular_scaler_v2.pkl")
TUNED_XGB = joblib.load("models/xgboost_tuned.pkl")'''
)

with open('src/api.py', 'w') as f:
    f.write(new_api)

print("API updated!")

API updated!


In [6]:
print("Day 15 Summary")
print("=" * 50)
print("\nCompleted:")
print("  ✓ XGBoost hyperparameter tuning")
print("  ✓ Best params: n_estimators=200, max_depth=4, lr=0.1")
print("  ✓ Tuned XGBoost test accuracy: 99.70% (+0.20%)")
print("  ✓ EfficientNet comparison — ResNet50 kept")
print("  ✓ Tuned model saved to models/xgboost_tuned.pkl")

Day 15 Summary

Completed:
  ✓ XGBoost hyperparameter tuning
  ✓ Best params: n_estimators=200, max_depth=4, lr=0.1
  ✓ Tuned XGBoost test accuracy: 99.70% (+0.20%)
  ✓ EfficientNet comparison — ResNet50 kept
  ✓ Tuned model saved to models/xgboost_tuned.pkl
